In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
pip install transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.8 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [3]:
import pandas as pd
import numpy as np

from transformers import AutoModel, AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback, AutoModelForSeq2SeqLM
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer, util

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# import wandb
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

# wandb.login(key=wandb_api_key)

# wandb.init(
#     project="24f3004524-t22026",
#     name="deberta-baseline-run1"
# )

In [5]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


# EDA

In [6]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [7]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [9]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

# Evaluation Metric

In [10]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
    
    score = 0.0
    for actual, pred in zip(labels, predictions):
        if actual == pred[0]:
            score += 1.0
        elif actual == pred[1]:
            score += 0.5
        elif actual == pred[2]:
            score += 1/3
            
    return {"map3": score / len(labels)}

# Milestone 2

In [11]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')

dataset = dataset.map(lambda x: {'combined_text': str(x['prompt']) + " " + str(x['A'])})

len(dataset[51]['combined_text'])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614

In [12]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

tokenizer.vocab_size

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522

In [13]:
tokenizer.sep_token_id

102

In [14]:
prompt_list = [str(text) for text in dataset['prompt']]

encoded = tokenizer(
                prompt_list, 
                padding='max_length', 
                truncation=True, 
                max_length=128, 
                return_tensors='pt'
)

encoded['input_ids'].shape

torch.Size([2000, 128])

In [15]:
model = AutoModel.from_pretrained('bert-base-uncased')

inputs = tokenizer(dataset[0]['prompt'], return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
last_hidden_state.shape

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])

In [16]:
cls_vector = last_hidden_state[0, 0, :]

sum_first_5 = cls_vector[:5].sum().item()
sum_first_5

-1.200096845626831

In [17]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text = "Light-ion fusion is a technique."
inputs_attn = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

tokens = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
fusion_idx = tokens.index('fusion')

attention_matrix = outputs_attn.attentions[-1][0, 0, :, :] 

attn_weight = attention_matrix[0, fusion_idx].item()
attn_weight

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.10247313976287842

In [18]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


prompt_emb = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
opt_b_emb = st_model.encode(dataset[0]['B'], convert_to_tensor=True)

sim_score = util.cos_sim(prompt_emb, opt_b_emb).item()
sim_score

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658098340034485

In [19]:



dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
letters = ['A', 'B', 'C', 'D', 'E']

tfidf_preds = []
minilm_preds = []
minilm_map3_score = 0.0


for row in dataset:
    prompt_text = str(row['prompt'])
    options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    actual_ans = str(row['answer']) 
    
    # --- PIPELINE 1: TF-IDF Cosine Similarity ---
    vec = TfidfVectorizer()
    tfidf_matrix = vec.fit_transform([prompt_text] + options)
    tfidf_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    
    tfidf_top3 = [letters[idx] for idx in np.argsort(tfidf_sim)[-3:][::-1]]
    tfidf_preds.append(tfidf_top3)
    
    # --- PIPELINE 2: MiniLM Embeddings ---
    prompt_emb = st_model.encode(prompt_text, convert_to_tensor=True)
    opt_embs = st_model.encode(options, convert_to_tensor=True)
    
    minilm_sim = util.cos_sim(prompt_emb, opt_embs).cpu().numpy().flatten()
    
    minilm_top3 = [letters[idx] for idx in np.argsort(minilm_sim)[-3:][::-1]]
    minilm_preds.append(minilm_top3)
    
    if actual_ans in minilm_top3:
        rank = minilm_top3.index(actual_ans) + 1
        minilm_map3_score += 1.0 / rank


final_minilm_map3 = minilm_map3_score / len(dataset)

divergence_count = sum(
    1 for i in range(len(dataset))
    if str(dataset[i]['answer']) not in tfidf_preds[i] 
    and str(dataset[i]['answer']) in minilm_preds[i]
)

final_minilm_map3


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.42308333333333487

In [20]:
divergence_count

564

In [21]:
from transformers import pipeline

zs_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_1 = dataset[1]
options = [str(row_1['A']), str(row_1['B']), str(row_1['C'])]

res_default = zs_pipe(str(row_1['prompt']), candidate_labels=options)
top_score = res_default['scores'][0]
top_score

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.4574522376060486

In [22]:
res_multi = zs_pipe(str(row_1['prompt']), candidate_labels=options, multi_label=True)

sum_default = sum(res_default['scores']) 
sum_multi = sum(res_multi['scores'])     

diff = abs(sum_default - sum_multi)
diff

0.999490372636501

In [23]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

row_0 = dataset[0]

gen_prompt = f"Question: {row_0['prompt']}. Is the correct answer A: {row_0['A']} or B: {row_0['B']}? Answer with just the letter A or B."


inputs = tokenizer(gen_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=5)

exact_string = tokenizer.decode(outputs[0], skip_special_tokens=True)
exact_string

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

'B'

In [24]:
# wandb.finish()

# Submission Cell

In [25]:
# submission = pd.DataFrame({
#     "ID": test["id"],
#     "Prediction": test_predictions
# })

# submission.to_csv("submission.csv", index=False)

# submission.head()